# 第13章 ミニGPTを鍛える：GPUで本格学習

漱石の小説7冊、100万字で、自分のGPTを育てる

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。

**このノートはGPUを使います。** メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから実行してください。


## 準備：GPUのランタイムにする


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("使う装置：", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

## 第12章のGPTを取ってくる


In [ ]:
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/minigpt.py
from minigpt import GPT
print("ミニGPTの設計図を読み込みました")

## 漱石の7作品を読み込む


In [ ]:
import re, math, urllib.request

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    text = urllib.request.urlopen(url).read().decode("cp932")
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

works = {
    "こころ": "773_ruby_5968",
    "坊っちゃん": "752_ruby_2438",
    "吾輩は猫である": "789_ruby_5639",
    "三四郎": "794_ruby_4237",
    "草枕": "776_ruby_6020",
    "夢十夜": "799_ruby_6024",
    "それから": "56143_ruby_50824",
}
texts = {}
for title, file_id in works.items():
    texts[title] = load_soseki(file_id)
    print(f"{title}：{len(texts[title]):,} 文字")

all_text = "\n\n".join(texts.values())
chars = sorted(set(all_text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
print(f"合計：{len(all_text):,} 文字、語彙：{V} 文字")

In [ ]:
def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

kokoro = texts["こころ"]
nk = int(len(kokoro) * 0.9)
kokoro_train = encode(kokoro[:nk])
kokoro_val = encode(kokoro[nk:])          # 記録表と同じ検証データ

others = [encode(t[:int(len(t) * 0.9)]) for title, t in texts.items() if title != "こころ"]
all_train = torch.cat([kokoro_train] + others)
print("『こころ』だけの訓練データ：", len(kokoro_train))
print("7作品の訓練データ：        ", len(all_train))

## 学習の道具をそろえる


In [ ]:
block_size = 128    # 何文字前まで見るか（窓の大きさ）
batch_size = 32     # 1回に何個の窓を学習するか

def get_batch(data, g):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,), generator=g)
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def val_loss(model, data):
    model.eval()
    starts = range(0, len(data) - block_size - 1, block_size)
    xs = torch.stack([data[i:i + block_size] for i in starts])
    ys = torch.stack([data[i + 1:i + block_size + 1] for i in starts])
    total = 0.0
    for j in range(0, len(xs), 64):
        _, loss = model(xs[j:j + 64].to(device), ys[j:j + 64].to(device))
        total += loss.item() * len(xs[j:j + 64])
    model.train()
    return total / len(xs)

def lr_at(step, max_lr, total, warmup=200):
    if step < warmup:
        return max_lr * step / warmup
    progress = (step - warmup) / (total - warmup)
    return max_lr * 0.5 * (1 + math.cos(math.pi * progress))

## 学習ループ（GPU対応版）


In [ ]:
def train(train_data, max_iters, lr=1e-3, seed=0, log_every=500):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = GPT(V, block_size=block_size, n_embd=128, n_head=4, n_layer=4, dropout=0.1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scaler = torch.amp.GradScaler(device, enabled=(device == "cuda"))
    history = []
    for step in range(max_iters + 1):
        for group in opt.param_groups:
            group["lr"] = lr_at(step, lr, max_iters)
        x, y = get_batch(train_data, g)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        if step % log_every == 0:
            vl = val_loss(model, kokoro_val)
            history.append((step, loss.item(), vl))
            print(f"ステップ {step:5d}：訓練 {loss.item():.3f}  検証（こころ）{vl:.3f}")
    return model, history

## 実験1：『こころ』だけで学習


In [ ]:
model_k, hist_k = train(kokoro_train, max_iters=3000)

## 実験2：7作品で学習


In [ ]:
model_all, hist_all = train(all_train, max_iters=5000)

## 2つの実験を比べる


In [ ]:
!pip install -q matplotlib-fontja

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

plt.figure(figsize=(6.5, 3.8))
for hist, label in [(hist_k, "『こころ』だけ"), (hist_all, "7作品")]:
    steps = [h[0] for h in hist]
    plt.plot(steps, [h[2] for h in hist], marker="o", label=label + "（検証）")
plt.axhline(3.177, color="gray", linestyle="--", label="第10章のMLP")
plt.xlabel("ステップ"); plt.ylabel("検証損失（こころ）")
plt.ylim(2.4, 4.5)
plt.legend(); plt.title("データを増やすと、同じモデルがもっと賢くなる")
plt.show()

## 書かせてみる


In [ ]:
def write(model, start, n=150, temperature=0.8, seed=0):
    torch.manual_seed(seed)
    model.eval()
    idx = encode(start)[None].to(device)
    out = model.generate(idx, max_new_tokens=n, temperature=temperature)
    return "".join(itos[i] for i in out[0].tolist())

for start in ["吾輩は", "先生は", "　山路を登りながら"]:
    print(write(model_all, start))
    print("-" * 40)

## 大事なモデルを保存する


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
save_dir = "/content/drive/MyDrive/tsukuru-llm"
os.makedirs(save_dir, exist_ok=True)
checkpoint = {
    "model": model_all.state_dict(),
    "chars": chars,
    "config": {"vocab_size": V, "block_size": block_size, "n_embd": 128, "n_head": 4, "n_layer": 4, "dropout": 0.1},
}
torch.save(checkpoint, f"{save_dir}/soseki_gpt.pt")
print("保存しました：", round(os.path.getsize(f"{save_dir}/soseki_gpt.pt") / 1e6, 1), "MB")

## ヘッドたちの役割分担を見る


In [ ]:
sample = "先生は奥さんの顔を見て、彼女に話しかけた。"
idx = encode(sample)[None].to(device)
model_all.eval()
with torch.no_grad():
    model_all(idx)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
heads = model_all.blocks[-1].attn.heads          # 最後の層の4つのヘッド
for h, ax in enumerate(axes):
    ax.imshow(heads[h].wei[0].float().cpu(), cmap="Oranges")
    ax.set_xticks(range(len(sample))); ax.set_xticklabels(list(sample), fontsize=7)
    ax.set_yticks(range(len(sample))); ax.set_yticklabels(list(sample), fontsize=7)
    ax.set_title(f"ヘッド {h}")
plt.tight_layout()
plt.show()

In [ ]:
block_size = 256
batch_size = 64

def train_big(train_data, max_iters, lr=6e-4, seed=0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = GPT(V, block_size=block_size, n_embd=384, n_head=6, n_layer=6, dropout=0.2).to(device)
    print("パラメータの数：", sum(p.numel() for p in model.parameters()))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scaler = torch.amp.GradScaler(device, enabled=(device == "cuda"))
    for step in range(max_iters + 1):
        for group in opt.param_groups:
            group["lr"] = lr_at(step, lr, max_iters)
        x, y = get_batch(train_data, g)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        if step % 500 == 0:
            print(f"ステップ {step:5d}：訓練 {loss.item():.3f}  検証（こころ）{val_loss(model, kokoro_val):.3f}")
    return model

model_big = train_big(all_train, max_iters=5000)